In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [3]:
#Data preprocessing
dataset = pd.read_csv('train.csv')

X_train = dataset.drop(columns=['Target'])
y_train = dataset['Target']

ones = np.ones((X_train.shape[0], 1))
X_train = np.column_stack([ones, X_train.values])

x0 = np.array([-1.47, 0, 0, 0, 0])

In [4]:
def f_sum(x, U, v):
    dots = U @ x
    return np.sum((1 - v) * dots + np.logaddexp(0, -dots))

def grad_f_eval(x, U, v):
    g = 1/(1+np.exp(-U@x))
    return U.T @ (g-v)

In [5]:
# Task c.)
def gradient_method_const(f, x0, U, v, c=3e-5, epsilon=1e-3, max_iters=3_000_000):
    x = x0.astype(float).copy()
    k = 0
    x_iter = [x.copy()]
    f_val = [f(x, U, v)]
    while k < max_iters:
        grad = U.T @ (1/(1+np.exp(-U@x)) - v)
        if np.linalg.norm(grad) <= epsilon:
            break
        x -= c * grad
        k += 1
        x_iter.append(x.copy())
        f_val.append(f(x, U, v))

    f_val_check = all(f_val[i] >= f_val[i+1] for i in range(len(f_val)-1))
    print(f"Hodnota ucelovej funkcie nerastie: {f_val_check}")
    print(f"Pocet iteracii: {k}")

    return x, x_iter

import time
for c in [3e-7, 3e-5, 3e-3]:
    print(f"\n--- c = {c} ---")
    start = time.time()
    vysl, _ = gradient_method_const(f_sum, x0, X_train, y_train.values,
                                     c=c)

    elapsed = time.time() - start
    print(f"Výsledok: {vysl}")
    print(f"Čas: {elapsed:.2f}s")

Hodnota ucelovej funkcie nerastie: True
Pocet iteracii: 2736859
Výsledok: [-1.47360963e+00  3.07175527e+00  1.01989855e+01  1.76905539e+01
  4.84667713e-03]

--- c = 3e-07 ---
Hodnota ucelovej funkcie nerastie: True
Pocet iteracii: 3000000
Výsledok: [-0.88390857  2.97133462  3.58735415  4.45600849  0.00750751]
Čas: 81.75s

--- c = 3e-05 ---
Hodnota ucelovej funkcie nerastie: True
Pocet iteracii: 2736859
Výsledok: [-1.47360963e+00  3.07175527e+00  1.01989855e+01  1.76905539e+01
  4.84667713e-03]
Čas: 71.24s

--- c = 0.003 ---
Hodnota ucelovej funkcie nerastie: False
Pocet iteracii: 3000000
Výsledok: [-8.27305581  8.23145812 22.59051432 48.10574825  0.40823205]
Čas: 88.47s


In [6]:
# Task d.)
def gradient_method_backtracking(f, x0, U, v, alpha=0.499, delta=0.8, epsilon=1e-3, max_iters=3_000_000):
    x = x0.astype(float).copy()
    k = 0

    x_iter = [x.copy()]
    l_min = np.inf
    l_max = np.nan

    while k < max_iters:
        grad = grad_f_eval(x, U, v)

        if np.linalg.norm(grad) <= epsilon:
            break

        d = -grad
        lambda_k = 0.2

        fx = f(x, U, v)
        dot = np.dot(grad, d) * alpha
        while f(x + lambda_k * d, U, v) > fx + dot * lambda_k:
            lambda_k *= delta

        if(k == 0):
            l_max = lambda_k

        x += lambda_k * d
        k += 1
        if (k % 10_000 == 0):
            print(k)

        l_min = min(l_min, lambda_k)

    print(f"Pocet iteracii: {k}")
    print(f"lambda_min = {l_min}")
    print(f"lambda_max = {l_max}")

    return (x, x_iter, l_min, l_max)

start = time.time()
vysl_2 = gradient_method_backtracking(f_sum, x0, X_train, y_train.values)
elapsed = time.time() - start
print(f"Čas: {elapsed:.2f}s")
print(f"f_1 ma minimum priblizne v bode {vysl_2[0]} \n")
l_max, l_min = vysl_2[2], vysl_2[3]

10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
Pocet iteracii: 631781
lambda_min = 5.575186299632669e-06
lambda_max = 6.9689828745408365e-06
Čas: 386.13s
f_1 ma minimum priblizne v bode [-1.47361099e+00  3.07175538e+00  1.01989574e+01  1.76906299e+01
  4.84665956e-03] 



In [7]:
# Tasks e.)
def golden_section(f, a, b, tol=1e-6):

    phi = (1 + np.sqrt(5))/2
    rho = phi - 1

    c1 = rho*a + (1 - rho)*b
    f1 = f(c1)
    c2 = (1 - rho)*a + rho*b
    f2 = f(c2)

    while abs(b - a) > tol:

        if f1 < f2:
            b = c2
            c2 = c1
            f2 = f1
            c1 = rho*a + (1 - rho)*b
            f1 = f(c1)
        else:
            a = c1
            c1 = c2
            f1 = f2
            c2 = (1 - rho)*a + rho*b
            f2 = f(c2)

    return (a+b)/2

def cauchy_method(f, x0, U, v, a, b, epsilon=1e-3, max_iters=3_000_000):
    x = x0.astype(float).copy()
    k = 0

    x_iter = [x.copy()]

    while k < max_iters:

        grad = grad_f_eval(x, U, v)
        d = -grad

        if np.linalg.norm(grad) <= epsilon:
            break

        def f_new(z):
            return f(x + z * d, U, v)

        lambda_k = golden_section(f_new, a, b)

        x = x + lambda_k * d
        x_iter.append(x.copy())

        k += 1

    print(f"Pocet iteracii: {k}")
    return x, x_iter
start = time.time()
vysl_3_1 = gradient_method_backtracking(f_sum, x0, X_train, y_train.values, 0.1*l_min, 10*l_max)
elapsed = time.time() - start
print(f"Čas: {elapsed:.2f}s")
print(f"f_1 ma minimum priblizne v bode {vysl_3_1[0]} \n")

start = time.time()
vysl_3_2 = gradient_method_backtracking(f_sum, x0, X_train, y_train.values, l_min, l_max)
elapsed = time.time() - start
print(f"Čas: {elapsed:.2f}s")
print(f"f_1 ma minimum priblizne v bode {vysl_3_2[0]} \n")

10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
640000
650000
660000
670000
680000
690000
700000
710000
720000
730000
740000
750000
760000
770000
780000
790000
800000
810000
820000
830000
840000
850000
860000
870000
880000
890000
900000
910000
920000
930000
940000
950000
960000
970000
980000
990000
1000000
1010000
1020000
1030000
1040000
1050000
1060000
1070000
1080000
1090000
1100000
1110000
1120000
1130000
1140000
1150000
1160000
1170000
1180000
1190000
1200000
1210000
1220000
1230000
1240000
1250000
1260000
1270000
1280000
1290000
1300000
1310000
1320000
1330000
1340000
1350000
1360000
1370000
1380000
1390

In [8]:


# Minimum
res = minimize(
    fun=lambda x: f_sum(x, X_train, y_train),
    x0=x0,
    method="BFGS"
)

# Results for c.) - e.)

#x_opt_ref = res.x
x_opt_ref = np.array([-1.47377220e+00,  3.07173693e+00,  1.01906655e+01,  1.77027043e+01, 4.84402783e-03])
print("Referenčné riešenie:", x_opt_ref)

# Constant step
# Number of iterations: 2736859
# Time: 1m 6.4s
x_1 = np.array([-1.47360963e+00,  3.07175527e+00,  1.01989855e+01,  1.76905539e+01, 4.84667713e-03])
print(f"c\t\t   : {x_1}, chyba: {np.linalg.norm(x_1 - x_opt_ref)}\n")
# print(f"c\t\t   : {vysl_1[0]}, chyba: {np.linalg.norm(vysl_1[0] - x_opt_ref)}\n")

# Approximately optimal step
# Number of iterations: 633840
# Time: 3m 28.0s
x_2 = np.array([-1.47361102e+00,  3.07175538e+00,  1.01989558e+01,  1.76906325e+01, 4.84665901e-03])
print(f"d\t\t   : {x_2}, chyba: {np.linalg.norm(x_2 - x_opt_ref)} \n")
# print(f"d\t\t   : {vysl_2[0]}, chyba: {np.linalg.norm(vysl_2[0] - x_opt_ref)} \n")

# Golden section on [0.1*l_min, 10*l_max]
# Number of iterations: 2472704
# Time: 4m 6.8s
x_3 = np.array([-1.47363523e+00,  3.07175410e+00, 1.01979473e+01,  1.76922987e+01, 4.84631001e-03])
print(f"e\t\t   : {x_3}, chyba: {np.linalg.norm(x_3 - x_opt_ref)} \n")
# print(f"e\t\t   : {vysl_3_1[0]}, chyba: {np.linalg.norm(vysl_3_1[0] - x_opt_ref)} \n")

# Golden section on [l_min, l_max]
# Number of iterations: 3000000 (max_iter)
# Time: 1m 42.0s
x_4 = np.array([-1.43329736e+00,  3.06269986e+00,  9.99405599e+00,  1.61372798e+01, 5.15234320e-03])
print(f"e\t\t   : {x_4}, chyba: {np.linalg.norm(x_4 - x_opt_ref)} \n")
# print(f"e\t\t   : {vysl_3_2[0]}, chyba: {np.linalg.norm(vysl_3_2[0] - x_opt_ref)} \n")


Referenčné riešenie: [-1.47377220e+00  3.07173693e+00  1.01906655e+01  1.77027043e+01
  4.84402783e-03]
c		   : [-1.47360963e+00  3.07175527e+00  1.01989855e+01  1.76905539e+01
  4.84667713e-03], chyba: 0.014726893512865089

d		   : [-1.47361102e+00  3.07175538e+00  1.01989558e+01  1.76906325e+01
  4.84665901e-03], chyba: 0.014645263932344917 

e		   : [-1.47363523e+00  3.07175410e+00  1.01979473e+01  1.76922987e+01
  4.84631001e-03], chyba: 0.012701188267173343 

e		   : [-1.43329736e+00  3.06269986e+00  9.99405599e+00  1.61372798e+01
  5.15234320e-03], chyba: 1.5782677659346225 

